# FlowGuard - EDA & Preprocessing
Explore data, run preprocessing pipeline.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.config import load_config
from src.utils.reproducibility import setup_reproducibility

config = load_config("configs/base.yaml")
setup_reproducibility(config)

## Quick EDA on Raw Data

In [ ]:
# Sample from UNSW for EDA
sample = pd.read_csv("data/raw/NF-UNSW-NB15-v3.csv", nrows=50000)
print(f"Shape: {sample.shape}")
print(f"\nColumns: {list(sample.columns)}")
print(f"\nLabel distribution:\n{sample['Label'].value_counts()}")
print(f"\nAttack types:\n{sample['Attack'].value_counts()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'PROTOCOL']):
    if col in sample.columns:
        sample[col].hist(ax=ax, bins=50)
        ax.set_title(col)
plt.tight_layout()
plt.savefig("results/plots/raw_distributions.png", dpi=100)
plt.show()

## Run Preprocessing Pipeline

In [ ]:
from src.data.preprocess import preprocess_dataset, create_combined_unlabeled, PreprocessingStats

processed_dir = config['data']['processed_dir']
os.makedirs(processed_dir, exist_ok=True)

stats = None
for ds_info in config['data']['datasets']:
    name = ds_info['name']
    raw_path = ds_info['raw_file']
    if not os.path.exists(raw_path) and os.path.exists(raw_path + '.gz'):
        raw_path = raw_path + '.gz'
    output_path = os.path.join(processed_dir, f"{name}.parquet")
    
    print(f"\nProcessing: {name}")
    try:
        df, stats = preprocess_dataset(raw_path, output_path, config, fit_stats=(stats is None), stats=stats)
        print(f"  -> {output_path}: {len(df):,} rows, {df.shape[1]} cols")
    except FileNotFoundError as e:
        print(f"  ERROR: {e}")

# Save stats
if stats:
    stats.save(os.path.join(processed_dir, "preprocessing_stats.npz"))

In [ ]:
create_combined_unlabeled(config)
print("Combined unlabeled dataset created.")

In [ ]:
from src.data.splits import generate_all_splits
generate_all_splits("configs/base.yaml")

In [ ]:
for name in ['unsw', 'bot', 'ton', 'cic']:
    path = f"data/processed/{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"{name}: {len(df):,} rows, {df.shape[1]} cols")
        print(f"  Label dist: {df['Label'].value_counts().to_dict()}")